# Chosen model: jhu-clsp/mmBERT-base (Kaggle Models: aisuko/mmbert-base)

<!-- TODO(team): rename this notebook/kernel to `<university>_<campus>_<team>_C1`
     (campus in SGS|HNI|MEL|MYS), e.g. `RMIT_SGS_myteam_C1`. Also update `id`/`title` in kernel-metadata.json. -->

**Challenge 1 - Multilingual prompt-attack classification (BENIGN / JAILBREAK / PROMPT_INJECTION).**

Approach
- Backbone: `jhu-clsp/mmBERT-base` (ModernBERT architecture, multilingual), full fine-tune with `AutoModelForSequenceClassification` (3 labels), loaded offline from the attached Kaggle Model.
- Input: `text` only. Long prompts are truncated at token level keeping the **head and the tail** (max_len 512 = head 380ish + tail 128), because injections often sit at the end of a long prompt.
- Imbalance: cross-entropy with **sqrt inverse-frequency class weights**.
- Recipe: 5-fold StratifiedGroupKFold (groups = normalised first 60 chars, avoids near-duplicate leakage), lr 3e-5, 3 epochs, batch 16, 10% linear warmup then linear decay, AdamW wd 0.01, grad-clip 1.0, fp16 autocast (T4/P100 have no bf16). Last epoch is used.
- Post-processing: **per-class log-probability bias** (B, J, PI) tuned on out-of-fold predictions to maximise macro-F1; test probabilities are averaged over the 5 fold models, then the bias is added.
- Cross-validated result (Modal L4, same recipe): **OOF macro-F1 = 0.9657** with tuned biases (B,J,PI) = (0, -1.7, 0.8).

Runtime: internet OFF. Expect roughly 5-6 min/epoch/fold on a T4 -> about 90 min for 5 folds x 3 epochs. Fold outputs are checkpointed to `/kaggle/working/folds/`, so a re-run resumes where it stopped.

In [ ]:
# ---- Configuration (edit here) -------------------------------------------------
MODEL_NAME_FOR_REPORT = "jhu-clsp/mmBERT-base"
N_FOLDS      = 5          # fold split is always N_FOLDS-way ...
RUN_FOLDS    = None       # ... but you may train only a subset, e.g. [0, 1] (None = all)
EPOCHS       = 3
LR           = 3e-5
BATCH_SIZE   = 16
MAX_LEN      = 512
TAIL_TOKENS  = 128        # tokens kept from the end of long prompts
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED         = 42
DEBUG_ROWS   = 0          # >0: tiny smoke run (subsample train/test, 1 epoch)
# Optional overrides (leave None for auto-discovery)
DATA_DIR_OVERRIDE  = None
MODEL_DIR_OVERRIDE = None
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]   # fixed order

In [ ]:
import os, re, glob, json, time, math, random, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import transformers, sklearn
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
# ModernBERT (mmBERT) needs transformers >= 4.48. Kaggle images normally ship a newer one.
_v = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
if _v < (4, 48):
    raise RuntimeError(
        f"transformers {transformers.__version__} is too old for mmBERT (ModernBERT needs >= 4.48). "
        "Internet is OFF so `pip install -U transformers` is not possible: attach a Kaggle dataset "
        "containing a newer transformers wheel and install it with `pip install --no-index --find-links <dir> transformers`."
    )
assert torch.cuda.is_available(), "Enable a GPU accelerator (T4 x2 or P100) in the notebook settings."
DEV = torch.device("cuda:0")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
USE_BF16 = torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability(0)[0] >= 8
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16   # T4/P100 -> fp16
print("AMP dtype:", AMP_DTYPE)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

In [ ]:
# ---- Locate data and model (auto-discovery, works whatever the mount layout / version number) ----
def find_data_dir():
    if DATA_DIR_OVERRIDE: return Path(DATA_DIR_OVERRIDE)
    for p in ["/kaggle/input/rmit-hackathon-2026", "/kaggle/input/competitions/rmit-hackathon-2026"]:
        if os.path.exists(f"{p}/train.csv"): return Path(p)
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    assert hits, "train.csv not found under /kaggle/input - attach the competition data"
    return Path(hits[0]).parent

def find_model_dir():
    if MODEL_DIR_OVERRIDE: return Path(MODEL_DIR_OVERRIDE)
    # Expected, e.g.: /kaggle/input/mmbert-base/transformers/default/1  (or .../models/aisuko/mmbert-base/transformers/default/1)
    cands = glob.glob("/kaggle/input/**/config.json", recursive=True)
    def score(c):
        low = c.lower().replace("_", "-")
        if "mmbert-base" in low: return 2
        if "mmbert" in low and "small" not in low: return 1
        return 0
    good = []
    for c in cands:
        s = score(c)
        if s == 0: continue
        try: cfg = json.load(open(c))
        except Exception: continue
        if cfg.get("model_type") == "modernbert" and cfg.get("hidden_size", 768) == 768:
            good.append((s, c))
    if not good:  # fallback: any ModernBERT base-size checkpoint
        for c in cands:
            try: cfg = json.load(open(c))
            except Exception: continue
            if cfg.get("model_type") == "modernbert" and cfg.get("hidden_size") == 768: good.append((0, c))
    assert good, f"mmBERT-base not found. config.json files seen: {cands[:10]}. Add the Kaggle Model aisuko/mmbert-base (Transformers/default)."
    good.sort(key=lambda t: (-t[0], t[1]))
    return Path(good[0][1]).parent

DATA_DIR, MODEL_DIR = find_data_dir(), find_model_dir()
print("DATA_DIR :", DATA_DIR, os.listdir(DATA_DIR))
print("MODEL_DIR:", MODEL_DIR, os.listdir(MODEL_DIR))

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
for d in (train, test): d["text"] = d["text"].fillna("").astype(str)
if DEBUG_ROWS:
    train = pd.concat([g.sample(min(len(g), DEBUG_ROWS), random_state=0) for _, g in train.groupby("label")]).reset_index(drop=True)
    test = test.head(DEBUG_ROWS).reset_index(drop=True); EPOCHS = 1
    print("DEBUG run:", len(train), "train rows")
label2id = {l: i for i, l in enumerate(LABELS)}
y_all = train["label"].map(label2id).to_numpy()
print(train.shape, test.shape, train["label"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# ---- Folds: same recipe as our local CV (StratifiedGroupKFold, group = normalised first 60 chars) ----
def group_key(t): return re.sub(r"\s+", " ", str(t).lower()).strip()[:60]
groups = train["text"].map(group_key)
fold_of = np.full(len(train), -1)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for k, (_, va) in enumerate(sgkf.split(train, train["label"], groups)): fold_of[va] = k
print(pd.crosstab(fold_of, train["label"]))

# ---- Tokenizer + head/tail truncation at token level ----
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
if tok.pad_token is None: tok.pad_token = tok.eos_token
probe = tok("hello world", add_special_tokens=True)["input_ids"]
plain = tok("hello world", add_special_tokens=False)["input_ids"]
pos = next(i for i in range(len(probe) - len(plain) + 1) if probe[i:i + len(plain)] == plain)
prefix, suffix = probe[:pos], probe[pos + len(plain):]
keep = MAX_LEN - len(prefix) - len(suffix)
head_n = min(keep, max(keep - TAIL_TOKENS, keep // 2))
print(f"special prefix={prefix} suffix={suffix} keep={keep} head={head_n} tail={keep - head_n}")

def encode(texts):
    out = []
    for ids in tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]:
        if len(ids) > keep: ids = ids[:head_n] + ids[len(ids) - (keep - head_n):]
        out.append(prefix + ids + suffix)
    return out

t0 = time.time()
ids_train, ids_test = encode(train["text"]), encode(test["text"])
print(f"tokenized in {time.time() - t0:.0f}s; mean len train {np.mean([len(x) for x in ids_train]):.0f}, truncated {np.mean([len(x) >= MAX_LEN for x in ids_train]):.1%}")
PAD_ID = tok.pad_token_id

def collate(seqs, ys=None):
    L = max(len(s) for s in seqs)
    ii = torch.full((len(seqs), L), PAD_ID, dtype=torch.long); am = torch.zeros((len(seqs), L), dtype=torch.long)
    for r, s in enumerate(seqs): ii[r, :len(s)] = torch.tensor(s); am[r, :len(s)] = 1
    b = {"input_ids": ii, "attention_mask": am}
    if ys is not None: b["labels"] = torch.tensor(ys, dtype=torch.long)
    return b

def train_batches(idx, bs, rng):
    """Length-grouped shuffling (like HF group_by_length): shuffle, sort within mega-chunks, shuffle batches."""
    idx = rng.permutation(idx); chunk = bs * 50; batches = []
    for s in range(0, len(idx), chunk):
        c = sorted(idx[s:s + chunk], key=lambda i: len(ids_train[i]))
        batches += [c[j:j + bs] for j in range(0, len(c), bs)]
    rng.shuffle(batches); return batches

@torch.no_grad()
def predict_probs(model, id_lists, bs=32):
    model.eval(); order = np.argsort([len(x) for x in id_lists], kind="stable"); outs = []
    for s in range(0, len(order), bs):
        b = collate([id_lists[i] for i in order[s:s + bs]]); b = {k: v.to(DEV) for k, v in b.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE): lg = model(**b).logits
        outs.append(lg.float().cpu().numpy())
    lg = np.concatenate(outs).astype(np.float64); lg -= lg.max(-1, keepdims=True)
    p = np.exp(lg); p /= p.sum(-1, keepdims=True)
    res = np.empty_like(p); res[order] = p      # un-sort back to the original row order
    return res

def macro_f1(y, pred, k=3):
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float); den = cm.sum(0) + cm.sum(1)
    return float(np.mean(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)))

In [ ]:
# ---- Train one fold (plain PyTorch loop: robust across transformers versions) ----
WORK = Path("/kaggle/working"); FOLD_DIR = WORK / "folds"; FOLD_DIR.mkdir(parents=True, exist_ok=True)

def load_model():
    kw = dict(num_labels=3, id2label=dict(enumerate(LABELS)), label2id=label2id, ignore_mismatched_sizes=True)
    try: m = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, attn_implementation="sdpa", **kw)
    except Exception as e:
        print("sdpa load failed, using default attention:", repr(e)[:120]); m = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, **kw)
    m.config.pad_token_id = PAD_ID
    return m.to(DEV)

def train_fold(fold):
    seed_all(SEED + fold); rng = np.random.RandomState(SEED + fold)
    tr = np.where(fold_of != fold)[0]; va = np.where(fold_of == fold)[0]
    counts = np.bincount(y_all[tr], minlength=3).astype(float)
    cw = torch.tensor((counts.max() / counts) ** 0.5, dtype=torch.float, device=DEV)   # sqrt class weights
    print(f"[fold {fold}] train={len(tr)} val={len(va)} class weights={cw.cpu().numpy().round(3)}")
    model = load_model()
    nd = ["bias", "norm"]
    params = [{"params": [p for n, p in model.named_parameters() if not any(k in n.lower() for k in nd)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(k in n.lower() for k in nd)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=LR, betas=(0.9, 0.999), eps=1e-8)
    steps_per_epoch = math.ceil(len(tr) / BATCH_SIZE); total = steps_per_epoch * EPOCHS; warm = int(WARMUP_RATIO * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.cuda.amp.GradScaler(enabled=(AMP_DTYPE == torch.float16))
    step = 0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); run = 0.0
        for bi, b_idx in enumerate(train_batches(tr, BATCH_SIZE, rng)):
            b = collate([ids_train[i] for i in b_idx], y_all[b_idx]); b = {k: v.to(DEV) for k, v in b.items()}
            lab = b.pop("labels")
            with torch.autocast("cuda", dtype=AMP_DTYPE): lg = model(**b).logits
            loss = torch.nn.functional.cross_entropy(lg.float(), lab, weight=cw)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1; run += loss.item()
            if (bi + 1) % 100 == 0: print(f"  ep{ep} step {bi + 1}/{steps_per_epoch} loss {run / (bi + 1):.4f}")
        pv = predict_probs(model, [ids_train[i] for i in va])
        print(f"[fold {fold}] epoch {ep + 1}/{EPOCHS} loss {run / steps_per_epoch:.4f} val macro-F1 {macro_f1(y_all[va], pv.argmax(-1)):.4f} ({time.time() - t0:.0f}s)")
    pt = predict_probs(model, ids_test)          # last epoch
    np.save(FOLD_DIR / f"oof{fold}.npy", pv); np.save(FOLD_DIR / f"oof_idx{fold}.npy", va); np.save(FOLD_DIR / f"test{fold}.npy", pt)
    del model, opt; torch.cuda.empty_cache()
    return pv, pt

In [ ]:
# ---- Run folds (resumable: finished folds are reloaded from /kaggle/working/folds) ----
folds_to_run = list(range(N_FOLDS)) if RUN_FOLDS is None else list(RUN_FOLDS)
oof = np.full((len(train), 3), np.nan); test_probs = []
T0 = time.time()
for f in folds_to_run:
    p = FOLD_DIR / f"test{f}.npy"
    if p.exists() and not DEBUG_ROWS and len(np.load(FOLD_DIR / f"oof_idx{f}.npy")) == (fold_of == f).sum():
        print(f"[fold {f}] found checkpoint, skipping training"); va = np.load(FOLD_DIR / f"oof_idx{f}.npy")
        oof[va] = np.load(FOLD_DIR / f"oof{f}.npy"); test_probs.append(np.load(p)); continue
    pv, pt = train_fold(f); oof[np.where(fold_of == f)[0]] = pv; test_probs.append(pt)
    print(f"elapsed {(time.time() - T0) / 60:.1f} min")
have = ~np.isnan(oof[:, 0])
print(f"\nOOF rows: {have.sum()}/{len(train)}  raw OOF macro-F1: {macro_f1(y_all[have], oof[have].argmax(-1)):.4f}")

In [ ]:
# ---- Tune per-class log-prob bias on OOF (B fixed at 0; grid over J, PI) ----
def tune_bias(logp, y, lo=-3.0, hi=3.0, step=0.1):
    grid = np.round(np.arange(lo, hi + 1e-9, step), 3); best, best_b = -1.0, np.zeros(3)
    for b1, b2 in itertools.product(grid, grid):
        b = np.array([0.0, b1, b2]); f = macro_f1(y, (logp + b).argmax(-1))
        if f > best + 1e-12 or (abs(f - best) <= 1e-12 and np.abs(b).sum() < np.abs(best_b).sum()): best, best_b = f, b
    return best, best_b

ph = np.log(np.clip(oof[have], 1e-9, 1)); yh = y_all[have]
tuned, bias = tune_bias(ph, yh)
print(f"tuned OOF macro-F1 {tuned:.4f}  biases (B,J,PI) = {bias.round(2).tolist()}   (reference run: 0.9657 with [0, -1.7, 0.8])")
pt_ = (ph + bias).argmax(-1)
for c, lab in enumerate(LABELS):
    tp = ((yh == c) & (pt_ == c)).sum(); print(f"  F1 {lab}: {2 * tp / max((yh == c).sum() + (pt_ == c).sum(), 1):.4f}")
lang = train["language"].to_numpy()[have]
print("  per-language macro-F1:", {lg: round(macro_f1(yh[lang == lg], pt_[lang == lg]), 4) for lg in sorted(set(lang))})

In [ ]:
# ---- Write submission: mean of fold test probs -> log + bias -> argmax ----
test_p = np.mean(test_probs, 0); assert test_p.shape == (len(test), 3)
pred = [LABELS[i] for i in (np.log(np.clip(test_p, 1e-9, 1)) + bias).argmax(-1)]
mapping = dict(zip(test["id"], pred))
sub = sample.copy() if not DEBUG_ROWS else pd.DataFrame({"id": test["id"]})
sub["label"] = sub["id"].map(mapping)
if not DEBUG_ROWS: assert sub["label"].notna().all() and len(sub) == 6000 and set(sub["label"]) <= set(LABELS)
sub[["id", "label"]].to_csv("/kaggle/working/submission.csv", index=False)
np.save(WORK / "oof_all.npy", oof); np.save(WORK / "test_probs_mean.npy", test_p)
print(sub["label"].value_counts(normalize=True).round(3).to_dict()); print(sub.head())